In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 1. Загрузка реальных предсказаний
pred_path = Path("predictions.csv")  # или "predictions.csv"
if not pred_path.exists():
    raise FileNotFoundError("Сначала положите реальный predictions.csv!")

df = pd.read_csv(pred_path)

# Фильтруем по зафиксированным сплитам
df_cal = df[df["split"] == "calib"].copy()
df_test = df[df["split"] == "test"].copy()

class_cols = ["prob_normal", "prob_bacterial", "prob_viral"]
probs_cal = df_cal[class_cols].values
y_cal = df_cal["true_label"].values

probs_test = df_test[class_cols].values
y_test = df_test["true_label"].values

print(f"Калибровочных снимков: {len(y_cal)}, Тестовых: {len(y_test)}")

# 2. Расчет порога калибровки q_hat (alpha = 0.10 или 0.05)
alpha = 0.10
n_cal = len(y_cal)

# Несоответствие: 1 - вероятность истинного класса
scores_cal = 1.0 - probs_cal[np.arange(n_cal), y_cal]

# Квантиль с поправкой Ангелопулоса
q_level = np.ceil((n_cal + 1) * (1 - alpha)) / n_cal
q_hat = np.quantile(scores_cal, min(1.0, q_level), method="higher")
print(f"Порог отсечки q_hat: {q_hat:.4f} (класс берем при P >= {1 - q_hat:.4f})")

# 3. Построение конформных множеств на тесте
# Класс входит в множество, если P >= 1 - q_hat
test_prediction_sets = (probs_test >= (1.0 - q_hat))

# Гарантируем непустоту (если все < порога, берем argmax)
for i in range(len(test_prediction_sets)):
    if not test_prediction_sets[i].any():
        test_prediction_sets[i, np.argmax(probs_test[i])] = True

# 4. Метрики для статьи
set_sizes = test_prediction_sets.sum(axis=1)
coverage = np.mean([test_prediction_sets[i, y_test[i]] for i in range(len(y_test))])

print(f"Эмпирическое покрытие (Coverage): {coverage:.2%} (Ожидалось >= {1 - alpha:.0%})")
print(f"Средний размер множества |C(x)|: {set_sizes.mean():.2f}")
print(f"Доля уверенных (|C| = 1): {np.mean(set_sizes == 1):.2%}")
print(f"Доля сомнительных (|C| >= 2): {np.mean(set_sizes >= 2):.2%}")

# 5. Сохранение файла с разбивкой для XAI (Grad-CAM)
df_test["set_size"] = set_sizes
df_test["in_set_normal"] = test_prediction_sets[:, 0]
df_test["in_set_bacterial"] = test_prediction_sets[:, 1]
df_test["in_set_viral"] = test_prediction_sets[:, 2]

df_test.to_csv("test_conformal_results.csv", index=False)
print("Файл outputs/test_conformal_results.csv готов для напарника!")

Калибровочных снимков: 1171, Тестовых: 879
Порог отсечки q_hat: 0.7977 (класс берем при P >= 0.2023)
Эмпирическое покрытие (Coverage): 91.70% (Ожидалось >= 90%)
Средний размер множества |C(x)|: 1.43
Доля уверенных (|C| = 1): 59.61%
Доля сомнительных (|C| >= 2): 40.39%
Файл outputs/test_conformal_results.csv готов для напарника!
